### 1. Importing Data and Libraries

In [1]:
!curl "https://api.mockaroo.com/api/20f2a030?count=1000&key=34c1ff90" > "Employee.csv"
!curl "https://api.mockaroo.com/api/bdb27dd0?count=1000&key=34c1ff90" > "departments.csv"
!curl "https://api.mockaroo.com/api/797b2da0?count=1000&key=34c1ff90" > "Salary.csv"

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 48086    0 48086    0     0  34782      0 --:--:--  0:00:01 --:--:-- 34794
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 33547    0 33547    0     0  15539      0 --:--:--  0:00:02 --:--:-- 15545
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 31333    0 31333    0     0  18113      0 --:--:--  0:00:01 --:--:-- 18122


In [2]:
import pandas as pd
import numpy as np
import os

import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
employees = pd.read_csv("/content/Employee.csv")
salaries = pd.read_csv("/content/Salary.csv")
departments = pd.read_csv("/content/departments.csv")


### 2. Database Schemas

In [4]:
employees_schema = """
create table employees (
	id INT,
	first_name VARCHAR(50),
	last_name VARCHAR(50),
	email VARCHAR(50),
	gender VARCHAR(50)
);
"""

salaries_schema = """
  create table salaries (
    employee_id VARCHAR(50),
    salary DECIMAL(8,2),
    hire_date DATE,
    years_of_experience INT,
    bonus_percentage DECIMAL(4,2)
  );
"""

departments_schema = """
  create table departments (
	employee_id VARCHAR(50),
	department_name VARCHAR(10),
	manager_id INT,
	office_location VARCHAR(50)
  );
"""


In [5]:
db_name = 'employee.db'
if os.path.exists(db_name):
    os.remove(db_name)
    print(f"Removed existing database '{db_name}'.")

In [6]:
import sqlite3
import pandas as pd
import os

COLUMN_DATA_TYPES = {
    'employees': {
        'id': 'INT',
        'first_name': 'VARCHAR(50)',
        'last_name': 'VARCHAR(50)',
        'email': 'VARCHAR(50)',
        'gender': 'VARCHAR(50)'
    },
    'salaries': {
        'employee_id': 'VARCHAR(50)',
        'salary': 'DECIMAL(8,2)',
        'hire_date': 'DATE',
        'years_of_experience': 'INT',
        'bonus_percentage': 'DECIMAL(4,2)'
    },
    'departments': {
        'employee_id': 'INT',
        'department_name': 'VARCHAR(100)',
        'manager_id': 'INT',
        'office_location': 'VARCHAR(100)'
    }
}

# --- Database setup ---
db_name = 'employee.db'
conn = None  # Initialize connection to None

try:
    # Establish a connection to the SQLite database
    conn = sqlite3.connect(db_name)
    cursor = conn.cursor()
    print(f"Database '{db_name}' created and connected successfully. ✅")

    # Create tables
    cursor.execute(employees_schema)
    cursor.execute(salaries_schema)
    cursor.execute(departments_schema)

    # --- Load data from CSV files into the tables using pandas ---
    csv_to_table_map = {
        '/content/Employee.csv': 'employees',
        '/content/Salary.csv': 'salaries',
        '/content/departments.csv': 'departments'
    }

    for csv_file, table_name in csv_to_table_map.items():
        if os.path.exists(csv_file):
            print(f"\nProcessing '{csv_file}' for table '{table_name}'...")

            # Read the CSV file into a pandas DataFrame
            df = pd.read_csv(csv_file)

            # 1. Get the expected schema for the current table
            expected_schema = COLUMN_DATA_TYPES[table_name]
            expected_cols = list(expected_schema.keys())

            # 2. Handle missing/extra columns
            # Drop columns from DataFrame that are not in the schema
            df = df[df.columns.intersection(expected_cols)]

            # Add any missing columns and fill with None (which becomes NULL in SQL)
            for col in expected_cols:
                if col not in df.columns:
                    df[col] = None

            # 3. Reorder columns to match the defined schema exactly
            df = df[expected_cols]

            # 4. Enforce data types
            for col, dtype in expected_schema.items():
                if 'datetime' in dtype:
                    # Use pd.to_datetime for date/time columns, coercing errors to NaT (Not a Time)
                    df[col] = pd.to_datetime(df[col], errors='coerce')
                else:
                    # Use astype for other columns, handling potential conversion errors
                    try:
                        df[col] = df[col].astype(dtype)
                    except (ValueError, TypeError) as e:
                        print(f"  - Warning: Could not convert column '{col}' to {dtype}. Error: {e}. Leaving as is.")


            # Use the to_sql method to insert the cleaned DataFrame
            df.to_sql(table_name, conn, if_exists='append', index=False)
            print(f"  -> Data from '{csv_file}' loaded into '{table_name}' table successfully.")
        else:
            print(f"Warning: '{csv_file}' not found. Skipping data load for '{table_name}'.")

    # Commit the changes to the database
    conn.commit()
    print("\nData committed to the database successfully. 🎉")

except sqlite3.Error as e:
    print(f"Database error: {e}")
except pd.errors.EmptyDataError as e:
    print(f"Pandas error: {e}. One of the CSV files might be empty.")
except KeyError as e:
    print(f"Schema definition error: A column is missing from the TABLE_DATA_TYPES dictionary: {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")
finally:
    # Close the connection if it was established
    if conn:
        conn.close()
        print("Database connection closed.")

Database 'employee.db' created and connected successfully. ✅

Processing '/content/Employee.csv' for table 'employees'...
  - Warning: Could not convert column 'id' to INT. Error: data type 'INT' not understood. Leaving as is.
  - Warning: Could not convert column 'first_name' to VARCHAR(50). Error: data type 'VARCHAR(50)' not understood. Leaving as is.
  - Warning: Could not convert column 'last_name' to VARCHAR(50). Error: data type 'VARCHAR(50)' not understood. Leaving as is.
  - Warning: Could not convert column 'email' to VARCHAR(50). Error: data type 'VARCHAR(50)' not understood. Leaving as is.
  - Warning: Could not convert column 'gender' to VARCHAR(50). Error: data type 'VARCHAR(50)' not understood. Leaving as is.
  -> Data from '/content/Employee.csv' loaded into 'employees' table successfully.

Processing '/content/Salary.csv' for table 'salaries'...
  - Warning: Could not convert column 'employee_id' to VARCHAR(50). Error: data type 'VARCHAR(50)' not understood. Leaving as 

### 3. Importing GenAI Libraries

In [7]:
from google import genai
from google.colab import userdata

In [8]:
genai_client = genai.Client(api_key=userdata.get('GOOGLE_API_KEY'))

### 4. Prompt Engineering

In [9]:
prompt = """
### **ROLE**

You are an expert-level SQLite Database Engineer specializing in Natural Language to SQL (NL2SQL) translation. Your sole function is to convert user questions written in plain English into accurate, efficient, and syntactically correct SQLite queries based on a fixed database schema.

-----

### **CONTEXT**

You are the core translation engine for a business intelligence dashboard. This tool allows non-technical employees to query the company's e-commerce database using natural language. The database dialect is always **SQLite**. Your responses will be executed directly on the database.

The database consists of the following three tables:

**`employees` table:**

```sql
create table employees (
	id INT,
	first_name VARCHAR(50),
	last_name VARCHAR(50),
	email VARCHAR(50),
	gender VARCHAR(50)
);
```

**`salaries` table:**

```sql
create table salaries (
    employee_id VARCHAR(50),
    salary DECIMAL(8,2),
    hire_date DATE,
    years_of_experience INT,
    bonus_percentage DECIMAL(4,2)
  );
```

**`departments` table:**

```sql
create table departments (
	employee_id VARCHAR(50),
	department_name VARCHAR(10),
	manager_id INT,
	office_location VARCHAR(50)
  );
```

-----

### **TASK**

Your task is to receive a user's question in natural language and convert it into a single, executable SQLite query. Follow these steps meticulously:

1.  **Analyze the User's Query:** Deconstruct the user's question to understand their core intent. Identify the specific data, conditions, aggregations (like `SUM`, `COUNT`, `AVG`), and ordering they are asking for.
2.  **Map to the Schema:** Map the entities from the user's query to the appropriate tables (`customers`, `products`, `orders`) and columns. Determine the necessary `JOIN` operations using `customers.customer_id` and `products.product_id` as foreign keys in the `orders` table.
3.  **Construct the SQLite Query:** Write a clean and efficient `SELECT` statement that is syntactically correct for SQLite. Ensure all table and column names are accurate.
4.  **Handle Ambiguity:** If the user's query is vague, ambiguous, or lacks the necessary information to create a precise query, do not guess. Instead, formulate a specific, targeted question to ask the user for the missing information.

-----

### **CONSTRAINTS**

  * **Read-Only Operations:** You must **ONLY** generate `SELECT` queries. Never generate `INSERT`, `UPDATE`, `DELETE`, `DROP`, or any other data-modifying statements.
  * **Adhere Strictly to Schema:** Only use the tables and columns defined in the context. Do not invent or assume the existence of any other tables or columns.
  * **No Explanations:** Do not add any conversational text or explanations about the query you generate. Your output must strictly follow the specified format.
  * **Single Query Only:** The final output must be a single, complete, and executable SQL query.
  * **Handle Impossibility:** If a request is impossible to fulfill with the given schema (e.g., "Which employee made the most sales?"), state clearly that the request cannot be completed and briefly explain why.

-----

### **EXAMPLES**

**Example 1: Simple Lookup**

  * **User Query:** "Show me all employees in the IT department"
  * **Expected Output:**
    ```json
    {
      "status": "success",
      "response": "SELECT e.first_name, e.last_name FROM employees e JOIN MOCK_DATA d ON e.id = d.employee_id WHERE d.department_name = 'IT';"
    }
    ```

**Example 2: Complex Join and Aggregation**

  * **User Query:** "Show employees with more than 5 years of experience and their salary"
  * **Expected Output:**
    ```json
    {
      "status": "success",
      "response": "SELECT e.first_name, e.last_name, s.salary FROM employees e JOIN salaries s ON e.id = s.employee_id WHERE s.years_of_experience > 5;"
    }
    ```

**Example 3: Ambiguous Query**

  * **User Query:** "Show me recent hires"
  * **Expected Output:**
    ```json
    {
      "status": "clarification_needed",
      "response": "Could you please define what 'recent' means? For example, 'in the last 7 days', 'this month', or 'since January 2025'."
    }
    ```

**Example 4: Impossible Query**

  * **User Query:** "Which department has the most sales?"
  * **Expected Output:**
    ```json
    {
      "status": "error",
      "response": "I cannot answer this question as the database does not contain information about sales."
    }
    ```

-----

### **OUTPUT FORMAT**

Your final response must be a single JSON object with two keys:

1.  `"status"`: A string with one of three possible values: `"success"`, `"clarification_needed"`, or `"error"`.
2.  `"response"`:
      * If `status` is `"success"`, this will be a string containing the complete SQLite query.
      * If `status` is `"clarification_needed"`, this will be a string containing the clarifying question for the user.
      * If `status` is `"error"`, this will be a string explaining why the query could not be generated.
"""

In [10]:
import json
def get_sql_query(genai_client, prompt, user_query):

  # https://www.geeksforgeeks.org/python/formatted-string-literals-f-strings-python/
  contents = f"""
  {prompt}

  Here's the user query in english you need to work on:
  {user_query}
  """
  response = genai_client.models.generate_content(model='gemini-2.5-flash', contents=contents)
  # print(response)

  # Access the usage_metadata attribute
  usage_metadata = response.usage_metadata

  # Print the different token counts
  print(f"Input Token Count: {usage_metadata.prompt_token_count}")
  print(f"Thoughts Token Count: {response.usage_metadata.thoughts_token_count}")
  print(f"Output Token Count: {usage_metadata.candidates_token_count}")
  print(f"Total Token Count: {usage_metadata.total_token_count}")

  output = json.loads(response.text.replace('```json', '').replace('```', ''))

  return output


In [11]:
import sqlite3
import pandas as pd

def execute_query(query, db_name='employee.db'):

    conn = None
    try:
        # Connect to the database
        conn = sqlite3.connect(db_name)
        cursor = conn.cursor()

        # Execute the query
        print(f"\nExecuting query on '{db_name}':\n{query}")
        cursor.execute(query)

        # Fetch all results
        results = cursor.fetchall()

        # Get column names from the cursor description
        columns = [description[0] for description in cursor.description]

        # Format results as a dataframe for easier use
        results_as_dict = [dict(zip(columns, row)) for row in results]
        results_df = pd.DataFrame(results_as_dict)

        print("Query executed successfully.")
        return results_df

    except sqlite3.Error as e:
        print(f"Database error executing query: {e}")
        return None
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return None
    finally:
        if conn:
            conn.close()

In [12]:
def text2sql(genai_client, prompt, user_query):
  output = get_sql_query(genai_client, prompt, user_query)
  if output['status'] == 'success':
    results = execute_query(output['response'])
    return results
  return output

### 5. LLM Invocation

In [13]:
text2sql(genai_client, prompt, "Show me the employee count by departments")

Input Token Count: 1307
Thoughts Token Count: 46
Output Token Count: 63
Total Token Count: 1416

Executing query on 'employee.db':
SELECT d.department_name, COUNT(e.id) AS employee_count FROM employees e JOIN departments d ON e.id = d.employee_id GROUP BY d.department_name;
Query executed successfully.


,department_name,employee_count
0,Finance,193
1,HR,198
2,IT,186
3,Marketing,197
4,Operations,226


In [14]:
text2sql(genai_client, prompt, "Show me the employee with their experience")

Input Token Count: 1307
Thoughts Token Count: 64
Output Token Count: 59
Total Token Count: 1430

Executing query on 'employee.db':
SELECT e.first_name, e.last_name, s.years_of_experience FROM employees e JOIN salaries s ON e.id = s.employee_id;
Query executed successfully.


,first_name,last_name,years_of_experience
0,Burtie,Funnell,11
1,Bobinette,Garlee,0
2,Vonni,Whorlton,23
3,Justino,Zorn,15
4,Bernice,Pierro,5
...,...,...,...
995,Ken,Idiens,3
996,Ed,Trimming,14
997,Ingra,Schirok,6
998,Farris,Guerreru,25
